# Sprint 1 — Data Acquisition and Environment Setup

Downloads and organizes the ASHRAE 90.1-2019 prototype building IDF files
used throughout this study, and prepares the EnergyPlus execution
environment.

**Building set (16 total):**
- 5 tested buildings — simulated under all prompt formats (F1-F4); their
  parameters are the ground truth for GSO / CV(RMSE) in later sprints.
- 11 pool buildings — never simulated; used only to construct independent,
  non-target-centred parameter ranges (Leave-One-Building-Out, Sprint 2).

**Source:** ANSI/ASHRAE/IES Standard 90.1-2019 Prototype Building Models,
energycodes.gov. (https://www.energycodes.gov/prototype-building-models)

**Climate:** Buffalo, NY (ASHRAE Climate Zone 5A) is the main-experiment
climate — the only one with a matching prototype IDF. Miami (1A) and
International Falls (7) EPW files are also prepared, but are applied to the
same Buffalo/CZ5A IDF; runs under these two are a weather-file stress test,
not an independent multi-climate DOE prototype validation.

**Reproducibility:** `environment_sprint1.json` records the software
environment and `MASTER_SEED`; `extraction_manifest.json` records, per
building, the source ZIP name, the selected internal IDF filename, and the
SHA-256 of every IDF and EPW file used downstream.

In [1]:
# S1-0  Setup
from google.colab import drive
drive.mount('/content/drive')

import os, re, subprocess, zipfile, hashlib

BASE    = '/content/drive/MyDrive/BEM-LLM'
DATA    = f'{BASE}/data'
ZIP_DIR = f'{DATA}/idf_referans/ashrae901_2019_prototypes'
REF     = f'{DATA}/idf_referans'
EP      = '/usr/local/EnergyPlus/energyplus'

MASTER_SEED = int(hashlib.sha256(b"BEM-LLM-v1").hexdigest(), 16) % (2**31)

os.makedirs(REF, exist_ok=True)

if not os.path.exists(EP):
    subprocess.run(['wget', '-q', '-O', '/tmp/ep.tar.gz',
        'https://github.com/NREL/EnergyPlus/releases/download/v23.2.0/'
        'EnergyPlus-23.2.0-7636e6b3e9-Linux-Ubuntu20.04-x86_64.tar.gz'])
    os.makedirs('/usr/local/EnergyPlus', exist_ok=True)
    subprocess.run(['tar', '-xzf', '/tmp/ep.tar.gz',
                    '-C', '/usr/local/EnergyPlus', '--strip-components=1'])

print(subprocess.run([EP, '--version'], capture_output=True, text=True).stdout.strip())
print(f'MASTER_SEED = {MASTER_SEED}')

Mounted at /content/drive
EnergyPlus, Version 23.2.0-7636e6b3e9
MASTER_SEED = 1542799867


In [2]:
# S1-A  Building lists
TESTED_BUILDINGS = [
    'OfficeMedium', 'SchoolPrimary', 'RetailStripmall',
    'ApartmentMidRise', 'RestaurantFastFood',
]
POOL_BUILDINGS = [
    'ApartmentHighRise', 'Hospital', 'HotelLarge', 'HotelSmall', 'OfficeLarge',
    'OfficeSmall', 'OutPatientHealthCare', 'RestaurantSitDown',
    'RetailStandalone', 'SchoolSecondary', 'Warehouse',
]
ALL_BUILDINGS = TESTED_BUILDINGS + POOL_BUILDINGS  # 16 total

def zip_path(name):
    return f'{ZIP_DIR}/ASHRAE901_{name}_STD2019.zip'

def idf_path(name):
    return f'{REF}/ASHRAE901_{name}_STD2019_Buffalo.idf'

missing_zips = [b for b in ALL_BUILDINGS if not os.path.exists(zip_path(b))]
print(f'{len(ALL_BUILDINGS) - len(missing_zips)}/{len(ALL_BUILDINGS)} source ZIPs found.')
for m in missing_zips:
    print('  missing:', zip_path(m))

16/16 source ZIPs found.


In [3]:
# S1-B  Extraction (Buffalo-climate IDF only, per building)
extracted = []
SELECTED_MEMBER = {}

for bldg in ALL_BUILDINGS:
    zp = zip_path(bldg)
    if not os.path.exists(zp):
        continue
    target = idf_path(bldg)
    with zipfile.ZipFile(zp, 'r') as zf:
        member_name = None
        for member in zf.namelist():
            if member.lower().endswith('.idf') and 'buffalo' in member.lower():
                member_name = member
                if not os.path.exists(target):
                    with zf.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
                break
        SELECTED_MEMBER[bldg] = member_name
    if os.path.exists(target):
        extracted.append(bldg)

print(f'{len(extracted)}/{len(ALL_BUILDINGS)} Buffalo IDF files ready in {REF}.')

16/16 Buffalo IDF files ready in /content/drive/MyDrive/BEM-LLM/data/idf_referans.


In [4]:
# S1-C  Verification
missing_idfs = [b for b in ALL_BUILDINGS if not os.path.exists(idf_path(b))]
if missing_idfs:
    print(f'WARNING: {len(missing_idfs)} IDF(s) still missing:')
    for m in missing_idfs:
        print('  -', idf_path(m))
else:
    print('All 16 IDF files verified present. Ready for Sprint 2.')

All 16 IDF files verified present. Ready for Sprint 2.


In [5]:
# S1-D  EPW climate files -- download if missing (5A main, 1A/7 validation)
WEATHER_DIR = f'{DATA}/epw_iklim'
os.makedirs(WEATHER_DIR, exist_ok=True)

CLIMATE_EPW = {
    '5A': ('5A_Buffalo.epw',
           'https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/'
           'USA/NY/USA_NY_Buffalo.Niagara.Intl.AP.725280_TMY3/'
           'USA_NY_Buffalo.Niagara.Intl.AP.725280_TMY3.epw'),
    '1A': ('1A_Miami.epw',
           'https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/'
           'USA/FL/USA_FL_Miami.Intl.AP.722020_TMY3/'
           'USA_FL_Miami.Intl.AP.722020_TMY3.epw'),
    '7':  ('7_InternationalFalls.epw',
           'https://energyplus-weather.s3.amazonaws.com/north_and_central_america_wmo_region_4/'
           'USA/MN/USA_MN_International.Falls.Intl.AP.727470_TMY3/'
           'USA_MN_International.Falls.Intl.AP.727470_TMY3.epw'),
}

def epw_path(zone):
    return f'{WEATHER_DIR}/{CLIMATE_EPW[zone][0]}'

for zone, (fname, url) in CLIMATE_EPW.items():
    path = epw_path(zone)
    if not os.path.exists(path):
        print(f'Downloading {zone} ({fname})...')
        result = subprocess.run(['wget', '-q', '-O', path, url], capture_output=True, text=True)
        if result.returncode != 0 or not os.path.exists(path) or os.path.getsize(path) < 1024:
            print(f'  WARNING: download failed for {zone}. Check URL manually: {url}')
            if os.path.exists(path):
                os.remove(path)

missing = [z for z in CLIMATE_EPW if not os.path.exists(epw_path(z))]
if missing:
    print(f'\nWARNING: {len(missing)} climate zone(s) still missing EPW: {missing}')
    for z in missing:
        print('  -', epw_path(z))
else:
    sizes = {z: os.path.getsize(epw_path(z)) for z in CLIMATE_EPW}
    for z, s in sizes.items():
        print(f'{z}: {CLIMATE_EPW[z][0]}  ({s/1024:.0f} KB)')
    print('\nAll 3 EPW files (5A, 1A, 7) verified present. Ready for Sprint 4 climate robustness runs.')

5A: 5A_Buffalo.epw  (1513 KB)
1A: 1A_Miami.epw  (1537 KB)
7: 7_InternationalFalls.epw  (1517 KB)

All 3 EPW files (5A, 1A, 7) verified present. Ready for Sprint 4 climate robustness runs.


In [6]:
# S1-E  Write environment.json (reproducibility record + single source of truth for MASTER_SEED)
import sys, platform, json

env_record = {
    'sprint': 'Sprint 1',
    'python_version': sys.version,
    'platform': platform.platform(),
    'energyplus_version': subprocess.run([EP, '--version'], capture_output=True, text=True).stdout.strip(),
    'master_seed': MASTER_SEED,
    'master_seed_derivation': 'int(sha256("BEM-LLM-v1"), 16) % 2**31',
}
with open(f'{DATA}/environment_sprint1.json', 'w') as f:
    json.dump(env_record, f, indent=2)
print(f'Saved: {DATA}/environment_sprint1.json')
print(json.dumps(env_record, indent=2))

Saved: /content/drive/MyDrive/BEM-LLM/data/environment_sprint1.json
{
  "sprint": "Sprint 1",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "energyplus_version": "EnergyPlus, Version 23.2.0-7636e6b3e9",
  "master_seed": 1542799867,
  "master_seed_derivation": "int(sha256(\"BEM-LLM-v1\"), 16) % 2**31"
}


In [7]:
# S1-F  Reproducibility manifest: SHA-256 for every IDF and EPW file, plus the
# ZIP source and the internal member selected from it. The 16 prototype ZIPs
# come from a single listing page (per-building direct links are not
# individually tracked), so source_listing_url is identical across buildings.
SOURCE_LISTING_URL = 'https://www.energycodes.gov/prototype-building-models'

def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = {'source_listing_url': SOURCE_LISTING_URL, 'buildings': {}, 'epw_files': {}}

for bldg in ALL_BUILDINGS:
    manifest['buildings'][bldg] = {
        'source_zip_name': os.path.basename(zip_path(bldg)),
        'selected_internal_idf_filename': SELECTED_MEMBER.get(bldg),
        'idf_sha256': sha256_of_file(idf_path(bldg)) if os.path.exists(idf_path(bldg)) else None,
    }

for zone, (fname, url) in CLIMATE_EPW.items():
    path = epw_path(zone)
    manifest['epw_files'][zone] = {
        'filename': fname,
        'source_url': url,
        'sha256': sha256_of_file(path) if os.path.exists(path) else None,
    }

with open(f'{DATA}/extraction_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Saved: {DATA}/extraction_manifest.json')
print(f'{len(manifest["buildings"])} building records, {len(manifest["epw_files"])} EPW records.')

Saved: /content/drive/MyDrive/BEM-LLM/data/extraction_manifest.json
16 building records, 3 EPW records.
